# Notebook 05 _ Modélisation prédictive énergétique


## Objectif

Ce notebook est consacré à la préparation et à l’évaluation des premiers
modèles de prévision énergétique à un horizon d’une heure.

Le jeu de données utilisé est celui produit dans le Notebook 04. Il contient
les variables climatiques, temporelles et historiques, ainsi que les trois
cibles suivantes :

- production photovoltaïque à une heure ;
- production éolienne à une heure ;
- demande électrique à une heure.

La démarche suivie comprend :

1. le chargement et le contrôle du dataset enrichi ;
2. la sélection rigoureuse des variables explicatives ;
3. la séparation chronologique en ensembles d’entraînement, de validation
   et de test ;
4. la construction de modèles de référence ;
5. l’évaluation à l’aide des métriques MAE, RMSE et R² ;
6. la sauvegarde des résultats et des modèles.

Aucun mélange aléatoire des observations ne sera effectué afin de respecter
la nature temporelle des données.


In [1]:
# 1. IMPORTATION DES BIBLIOTHÈQUES

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

print("Bibliothèques importées.")

Bibliothèques importées.


In [2]:
# 2. DÉFINITION DES CHEMINS

PROJECT_DIR = Path.cwd().parent

PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"
FIGURES_DIR = PROJECT_DIR / "figures"
RESULTS_DIR = PROJECT_DIR / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = PROCESSED_DATA_DIR / "energy_features_1h.csv"

print("Projet :", PROJECT_DIR)
print("Dataset :", DATA_FILE)
print("Le fichier existe :", DATA_FILE.exists())

Projet : c:\Users\celes\Desktop\PFE
Dataset : c:\Users\celes\Desktop\PFE\data\processed\energy_features_1h.csv
Le fichier existe : True


In [3]:
# 3. CHARGEMENT DU DATASET

model_df = pd.read_csv(
    DATA_FILE,
    parse_dates=["Time"]
)

print("Dimensions :", model_df.shape)
print("Date minimale :", model_df["Time"].min())
print("Date maximale :", model_df["Time"].max())

display(model_df.head())

Dimensions : (313620, 118)
Date minimale : 2019-01-08 00:00:00
Date maximale : 2021-12-31 22:55:00


,Time,Season,DHI,DNI,GHI,Wind_speed,Humidity,Temperature,PV_production,Wind_production,Electric_demand,Year,Month,Day,Hour,Minute,Weekday,Is_weekend,target_PV_production_1h,target_Wind_production_1h,target_Electric_demand_1h,Hour_decimal,Hour_sin,Hour_cos,Weekday_sin,Weekday_cos,Day_of_year,Year_sin,Year_cos,PV_production_lag_5min,PV_production_lag_1h,PV_production_lag_6h,PV_production_lag_24h,PV_production_lag_7d,Wind_production_lag_5min,Wind_production_lag_1h,Wind_production_lag_6h,Wind_production_lag_24h,Wind_production_lag_7d,Electric_demand_lag_5min,Electric_demand_lag_1h,Electric_demand_lag_6h,Electric_demand_lag_24h,Electric_demand_lag_7d,GHI_lag_5min,GHI_lag_1h,GHI_lag_6h,GHI_lag_24h,GHI_lag_7d,Wind_speed_lag_5min,Wind_speed_lag_1h,Wind_speed_lag_6h,Wind_speed_lag_24h,Wind_speed_lag_7d,Temperature_lag_5min,Temperature_lag_1h,Temperature_lag_6h,Temperature_lag_24h,Temperature_lag_7d,Humidity_lag_5min,Humidity_lag_1h,Humidity_lag_6h,Humidity_lag_24h,Humidity_lag_7d,PV_production_rolling_mean_1h,PV_production_rolling_std_1h,PV_production_rolling_mean_6h,PV_production_rolling_std_6h,PV_production_rolling_mean_24h,PV_production_rolling_std_24h,Wind_production_rolling_mean_1h,Wind_production_rolling_std_1h,Wind_production_rolling_mean_6h,Wind_production_rolling_std_6h,Wind_production_rolling_mean_24h,Wind_production_rolling_std_24h,Electric_demand_rolling_mean_1h,Electric_demand_rolling_std_1h,Electric_demand_rolling_mean_6h,Electric_demand_rolling_std_6h,Electric_demand_rolling_mean_24h,Electric_demand_rolling_std_24h,GHI_rolling_mean_1h,GHI_rolling_std_1h,GHI_rolling_mean_6h,GHI_rolling_std_6h,GHI_rolling_mean_24h,GHI_rolling_std_24h,Wind_speed_rolling_mean_1h,Wind_speed_rolling_std_1h,Wind_speed_rolling_mean_6h,Wind_speed_rolling_std_6h,Wind_speed_rolling_mean_24h,Wind_speed_rolling_std_24h,Temperature_rolling_mean_1h,Temperature_rolling_std_1h,Temperature_rolling_mean_6h,Temperature_rolling_std_6h,Temperature_rolling_mean_24h,Temperature_rolling_std_24h,Humidity_rolling_mean_1h,Humidity_rolling_std_1h,Humidity_rolling_mean_6h,Humidity_rolling_std_6h,Humidity_rolling_mean_24h,Humidity_rolling_std_24h,PV_production_diff_5min,PV_production_diff_1h,Wind_production_diff_5min,Wind_production_diff_1h,Electric_demand_diff_5min,Electric_demand_diff_1h,GHI_diff_5min,GHI_diff_1h,Wind_speed_diff_5min,Wind_speed_diff_1h,Temperature_diff_5min,Temperature_diff_1h
0,2019-01-08 00:00:00,1,0.000,0.000,0.000,2.920,77.232,9.240,0,814,21490,2019,1,8,0,0,1,0,0.000,658.000,20600.000,0.000,0.000,1.000,0.782,0.623,8,0.137,0.991,0.000,75.000,75.000,0.000,0.000,824.000,916.000,1609.000,2848.000,2810.000,21596.000,22972.000,28665.000,21115.000,22216.000,0.000,0.000,0.000,0.000,0.000,2.940,2.900,2.640,3.420,2.880,9.220,9.220,9.960,7.920,1.820,77.314,77.226,76.950,80.290,56.036,68.750,21.651,73.958,8.839,1971.538,2791.243,848.333,28.637,1005.708,213.614,1282.896,729.465,22291.833,453.617,26018.667,2285.065,24016.250,2738.828,0.000,0.000,0.000,0.000,89.156,133.931,2.900,0.015,2.800,0.084,2.667,0.272,9.240,0.012,9.404,0.203,10.145,2.145,77.160,0.071,77.566,0.455,74.139,6.728,0.000,-75.000,-10.000,-102.000,-106.000,-1482.000,0.000,0.000,-0.020,0.020,0.020,0.020
1,2019-01-08 00:05:00,1,0.000,0.000,0.000,2.920,77.124,9.260,0,807,21411,2019,1,8,0,5,1,0,0.000,667.000,20514.000,0.083,0.022,1.000,0.782,0.623,8,0.137,0.991,0.000,75.000,75.000,0.000,0.000,814.000,876.000,1570.000,2891.000,2862.000,21490.000,22859.000,28613.000,21050.000,22106.000,0.000,0.000,0.000,0.000,0.000,2.920,2.900,2.640,3.400,2.880,9.240,9.260,9.940,7.920,1.820,77.232,77.034,77.040,80.290,56.036,62.500,29.194,72.917,12.412,1971.538,2791.243,839.833,20.788,994.667,202.234,1275.833,724.086,22168.333,453.345,25919.014,2324.150,24017.552,2737.532,0.000,0.000,0.000,0.000,89.156,133.931,2.902,0.016,2.804,0.083,2.666,0.269,9.242,0.010,9.394,0.193,10.150,2.142,77.161,0.071,77.570,0.451,74.129,6.721,0.000,-75.000,-7.000,-69.000,-79.000,-1448.000,0.000,0.000,0.000,0.020,0.020,0.000
2,2019-01-08 00

# 4. Contrôle de la qualité des données

Avant la modélisation, le dataset est contrôlé une dernière fois.

Les vérifications concernent :

- les valeurs manquantes ;
- les valeurs infinies ;
- les dates dupliquées ;
- l’ordre chronologique ;
- la présence des trois variables cibles.


In [4]:
# 4. CONTRÔLES DE QUALITÉ

target_columns = [
    "target_PV_production_1h",
    "target_Wind_production_1h",
    "target_Electric_demand_1h"
]

numeric_df = model_df.select_dtypes(include="number")

quality_checks = pd.DataFrame({
    "Contrôle": [
        "Valeurs manquantes",
        "Valeurs infinies",
        "Dates dupliquées",
        "Ordre chronologique",
        "Nombre d'observations",
        "Nombre de colonnes",
        "Cibles présentes"
    ],
    "Résultat": [
        int(model_df.isna().sum().sum()),
        int(np.isinf(numeric_df.to_numpy()).sum()),
        int(model_df["Time"].duplicated().sum()),
        model_df["Time"].is_monotonic_increasing,
        model_df.shape[0],
        model_df.shape[1],
        all(column in model_df.columns for column in target_columns)
    ]
})

display(quality_checks)

,Contrôle,Résultat
0,Valeurs manquantes,0
1,Valeurs infinies,0
2,Dates dupliquées,0
3,Ordre chronologique,True
4,Nombre d'observations,313620
5,Nombre de colonnes,118
6,Cibles présentes,True


# 5. Sélection des variables

Avant de construire les modèles prédictifs, il est nécessaire de distinguer :

- les variables explicatives (features) utilisées pour l'apprentissage ;
- les variables cibles (targets) que les modèles devront prédire.

Les variables de date, les variables cibles et les informations susceptibles
de provoquer une fuite de données (data leakage) sont exclues des variables
explicatives.


In [5]:
# 5. SÉLECTION DES VARIABLES

# Variables cibles
target_columns = [
    "target_PV_production_1h",
    "target_Wind_production_1h",
    "target_Electric_demand_1h"
]

# Colonnes à exclure des variables explicatives
excluded_columns = [
    "Time",
    *target_columns
]

# Variables explicatives
feature_columns = [
    col for col in model_df.columns
    if col not in excluded_columns
]

print("Nombre de variables explicatives :", len(feature_columns))
print("Nombre de variables cibles :", len(target_columns))

display(feature_columns[:20])

Nombre de variables explicatives : 114
Nombre de variables cibles : 3


['Season',
 'DHI',
 'DNI',
 'GHI',
 'Wind_speed',
 'Humidity',
 'Temperature',
 'PV_production',
 'Wind_production',
 'Electric_demand',
 'Year',
 'Month',
 'Day',
 'Hour',
 'Minute',
 'Weekday',
 'Is_weekend',
 'Hour_decimal',
 'Hour_sin',
 'Hour_cos']

# 6. Séparation des variables

Les variables explicatives et les variables cibles sont séparées afin de
préparer les données pour la phase d'apprentissage.

Cette séparation sera utilisée pour les trois modèles de prévision développés
dans ce projet.


In [6]:
# 6. SÉPARATION FEATURES TARGETS

X = model_df[feature_columns]

y_pv = model_df["target_PV_production_1h"]

y_wind = model_df["target_Wind_production_1h"]

y_demand = model_df["target_Electric_demand_1h"]

print("Dimensions des variables explicatives :", X.shape)

print("PV :", y_pv.shape)

print("Wind :", y_wind.shape)

print("Demand :", y_demand.shape)

Dimensions des variables explicatives : (313620, 114)
PV : (313620,)
Wind : (313620,)
Demand : (313620,)


# 7. Découpage chronologique des données

Contrairement aux jeux de données classiques, les séries temporelles ne doivent
pas être mélangées aléatoirement.

Le découpage est réalisé chronologiquement afin de respecter l'ordre temporel
des observations.

Les proportions retenues sont :

- 70 % pour l'entraînement ;
- 15 % pour la validation ;
- 15 % pour le test.


In [7]:
# 7. DÉCOUPAGE CHRONOLOGIQUE

n = len(model_df)

train_end = int(0.70 * n)

valid_end = int(0.85 * n)

print("Nombre total :", n)

print("Fin Train :", train_end)

print("Fin Validation :", valid_end)

Nombre total : 313620
Fin Train : 219534
Fin Validation : 266577


In [8]:
# 7.1 TRAIN VALIDATION TEST

X_train = X.iloc[:train_end]

X_valid = X.iloc[train_end:valid_end]

X_test = X.iloc[valid_end:]

y_train_pv = y_pv.iloc[:train_end]
y_valid_pv = y_pv.iloc[train_end:valid_end]
y_test_pv = y_pv.iloc[valid_end:]

y_train_wind = y_wind.iloc[:train_end]
y_valid_wind = y_wind.iloc[train_end:valid_end]
y_test_wind = y_wind.iloc[valid_end:]

y_train_demand = y_demand.iloc[:train_end]
y_valid_demand = y_demand.iloc[train_end:valid_end]
y_test_demand = y_demand.iloc[valid_end:]

print("=" * 60)

print("Train :", X_train.shape)

print("Validation :", X_valid.shape)

print("Test :", X_test.shape)

Train : (219534, 114)
Validation : (47043, 114)
Test : (47043, 114)


# 8. Normalisation des variables

Les variables explicatives sont normalisées à l'aide de la méthode
`StandardScaler`.

La normalisation est ajustée uniquement sur les données d'entraînement puis
appliquée aux ensembles de validation et de test afin d'éviter toute fuite
d'information (data leakage).

Cette étape est particulièrement importante pour les modèles linéaires et les
réseaux de neurones.


In [9]:
# 8. NORMALISATION DES VARIABLES

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_valid_scaled = scaler.transform(X_valid)

X_test_scaled = scaler.transform(X_test)

print("Train :", X_train_scaled.shape)
print("Validation :", X_valid_scaled.shape)
print("Test :", X_test_scaled.shape)

Train : (219534, 114)
Validation : (47043, 114)
Test : (47043, 114)


# 9. Modèle de référence : Régression linéaire

La régression linéaire est utilisée comme modèle de référence (baseline).

Bien que relativement simple, elle permet d'établir un premier niveau de
performance qui servira de comparaison avec les modèles plus avancés étudiés
par la suite.


In [10]:
# 9. MODÈLE BASELINE : RÉGRESSION LINÉAIRE (PV)

linear_model_pv = LinearRegression()

linear_model_pv.fit(
    X_train_scaled,
    y_train_pv
)

print("Modèle entraîné avec succès.")

Modèle entraîné avec succès.


# 10. Évaluation du modèle de référence

Les performances du modèle sont évaluées sur l'ensemble de test à l'aide des
métriques suivantes :

- MAE (Mean Absolute Error) ;
- RMSE (Root Mean Squared Error) ;
- Coefficient de détermination (R²).

Ces indicateurs permettront de comparer objectivement les différents modèles
développés dans cette étude.


In [11]:
# 10. PRÉDICTIONS

y_pred_pv = linear_model_pv.predict(X_test_scaled)

In [12]:
# 10.1 ÉVALUATION

mae = mean_absolute_error(
    y_test_pv,
    y_pred_pv
)

rmse = np.sqrt(
    mean_squared_error(
        y_test_pv,
        y_pred_pv
    )
)

r2 = r2_score(
    y_test_pv,
    y_pred_pv
)

print("=" * 50)
print("Régression linéaire - Production photovoltaïque")
print("=" * 50)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

Régression linéaire - Production photovoltaïque
MAE  : 463.30
RMSE : 672.04
R²   : 0.9786


## Interprétation des résultats

Le modèle de régression linéaire obtient un coefficient de détermination (R²)
de 0,9786, indiquant qu'il explique près de 98 % de la variabilité de la
production photovoltaïque.

Les erreurs MAE et RMSE restent relativement faibles au regard de l'amplitude
de la variable cible.

Ces résultats montrent que les variables explicatives construites lors de la
phase de Feature Engineering capturent efficacement les principaux facteurs
influençant la production photovoltaïque.

La régression linéaire constitue ainsi une excellente référence (baseline) qui
sera comparée à un modèle d'ensemble plus performant dans les sections
suivantes.


# 11. Modèle avancé : HistGradientBoostingRegressor

Afin d'améliorer les performances obtenues avec la régression linéaire, un
modèle d'ensemble basé sur des arbres de décision est entraîné.

Le modèle HistGradientBoostingRegressor est particulièrement adapté aux jeux de
données volumineux et permet de capturer des relations non linéaires entre les
variables climatiques, temporelles et énergétiques.


In [13]:
# 11. HISTGRADIENTBOOSTINGREGRESSOR

boosting_model_pv = HistGradientBoostingRegressor(
    random_state=42
)

boosting_model_pv.fit(
    X_train,
    y_train_pv
)

print("Modèle HistGradientBoosting entraîné.")

Modèle HistGradientBoosting entraîné.


In [14]:
# 11.1 PRÉDICTIONS

y_pred_boost = boosting_model_pv.predict(X_test)

In [15]:
# 11.2 ÉVALUATION

mae_boost = mean_absolute_error(
    y_test_pv,
    y_pred_boost
)

rmse_boost = np.sqrt(
    mean_squared_error(
        y_test_pv,
        y_pred_boost
    )
)

r2_boost = r2_score(
    y_test_pv,
    y_pred_boost
)

print("="*50)
print("HistGradientBoosting - Production photovoltaïque")
print("="*50)

print(f"MAE  : {mae_boost:.2f}")
print(f"RMSE : {rmse_boost:.2f}")
print(f"R²   : {r2_boost:.4f}")

HistGradientBoosting - Production photovoltaïque
MAE  : 269.95
RMSE : 470.21
R²   : 0.9895


## Interprétation des résultats

Le modèle HistGradientBoostingRegressor améliore significativement les
performances obtenues avec la régression linéaire.

Les erreurs de prédiction diminuent tandis que le coefficient de détermination
atteint 0,9895, ce qui indique que près de 99 % de la variabilité de la
production photovoltaïque est expliquée par le modèle.

Cette amélioration montre que les relations entre les variables climatiques,
temporelles et la production photovoltaïque sont essentiellement non linéaires.
Le modèle HistGradientBoosting est donc retenu comme modèle de référence
principal pour cette variable.


# 12. Fonction générale d'entraînement et d'évaluation

Afin d'éviter la duplication du code et de faciliter la comparaison entre les
différents modèles, une fonction générique est définie.

Cette fonction entraîne un modèle, réalise les prédictions sur l'ensemble de
test et calcule automatiquement les principales métriques de performance.

Les résultats sont ensuite stockés dans une structure de données qui sera
utilisée pour construire un tableau comparatif.


In [16]:
validation_results = []

In [17]:
# 12. FONCTION GÉNÉRALE D'ENTRAÎNEMENT ET D'ÉVALUATION

def evaluate_model(
    target_name,
    model_name,
    model,
    X_train_data,
    X_validation_data,
    y_train_data,
    y_validation_data
):
    """
    Entraîne un modèle sur l'ensemble d'entraînement,
    réalise les prédictions sur l'ensemble de validation
    et calcule les métriques MAE, RMSE et R².
    """

    # Entraînement
    model.fit(
        X_train_data,
        y_train_data
    )

    # Prédictions sur la validation
    predictions = model.predict(
        X_validation_data
    )

    # Calcul des métriques
    mae = mean_absolute_error(
        y_validation_data,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_validation_data,
            predictions
        )
    )

    r2 = r2_score(
        y_validation_data,
        predictions
    )

    # Enregistrement des résultats
    validation_results.append({
        "Cible": target_name,
        "Modèle": model_name,
        "MAE_validation": mae,
        "RMSE_validation": rmse,
        "R2_validation": r2
    })

    print("=" * 65)
    print("Cible   :", target_name)
    print("Modèle  :", model_name)
    print("-" * 65)
    print(f"MAE validation  : {mae:.2f}")
    print(f"RMSE validation : {rmse:.2f}")
    print(f"R² validation   : {r2:.4f}")

    return model, predictions

# 13. Baseline naïve saisonnière

Avant de comparer les algorithmes de Machine Learning, une méthode simple de référence est évaluée.

Les données étant espacées de 5 minutes, une journée correspond à 288 observations. La prévision naïve saisonnière estime donc la valeur future à partir de la valeur observée au même instant la veille :

\[
\hat{y}_t = y_{t-288}
\]

Cette baseline permet de vérifier que les modèles entraînés apportent un gain réel par rapport à une règle de prévision très simple.


In [18]:
#13. BASELINE NAÏVE SAISONNIÈRE JOURNALIÈRE

SEASON_LENGTH = 24 * 12  # 24 heures × 12 observations de 5 minutes = 288


def evaluate_seasonal_naive(
    target_name,
    full_target,
    validation_index,
    season_length=SEASON_LENGTH
):
    """
    Évalue une prévision naïve saisonnière sur la validation.

    Pour chaque observation, la prédiction correspond à la valeur de la
    cible observée au même instant la veille. Le décalage est appliqué sur
    la série complète afin que les premières observations de validation
    puissent utiliser l'historique de l'ensemble d'entraînement.
    """

    seasonal_predictions = (
        full_target
        .shift(season_length)
        .loc[validation_index]
    )

    if seasonal_predictions.isna().any():
        raise ValueError(
            "La baseline contient des valeurs manquantes. "
            "Vérifier la longueur de saison et le découpage temporel."
        )

    y_validation = full_target.loc[validation_index]

    mae = mean_absolute_error(y_validation, seasonal_predictions)
    rmse = np.sqrt(mean_squared_error(y_validation, seasonal_predictions))
    r2 = r2_score(y_validation, seasonal_predictions)

    validation_results.append({
        "Cible": target_name,
        "Modèle": "Naïf saisonnier journalier",
        "MAE_validation": mae,
        "RMSE_validation": rmse,
        "R2_validation": r2
    })

    print("=" * 65)
    print("Cible   :", target_name)
    print("Modèle  : Naïf saisonnier journalier")
    print("-" * 65)
    print(f"MAE validation  : {mae:.2f}")
    print(f"RMSE validation : {rmse:.2f}")
    print(f"R² validation   : {r2:.4f}")

    return seasonal_predictions


# 14. Comparaison des modèles pour la production photovoltaïque

La régression linéaire et le modèle HistGradientBoosting sont réentraînés à
l'aide de la fonction générale.

Les performances sont désormais mesurées sur l'ensemble de validation.
L'ensemble de test reste réservé à l'évaluation finale du modèle retenu.

La régression linéaire utilise les variables normalisées, tandis que le modèle
HistGradientBoosting utilise les variables originales, car les modèles fondés
sur des arbres ne nécessitent pas de normalisation.


In [19]:
pred_valid_naive_pv = evaluate_seasonal_naive(
    target_name="Production photovoltaïque",
    full_target=y_pv,
    validation_index=y_valid_pv.index
)


Cible   : Production photovoltaïque
Modèle  : Naïf saisonnier journalier
-----------------------------------------------------------------
MAE validation  : 523.48
RMSE validation : 1030.39
R² validation   : 0.9587


In [20]:
linear_model_pv, pred_valid_linear_pv = evaluate_model(
    target_name="Production photovoltaïque",
    model_name="Régression linéaire",
    model=LinearRegression(),
    X_train_data=X_train_scaled,
    X_validation_data=X_valid_scaled,
    y_train_data=y_train_pv,
    y_validation_data=y_valid_pv
)

Cible   : Production photovoltaïque
Modèle  : Régression linéaire
-----------------------------------------------------------------
MAE validation  : 508.67
RMSE validation : 708.11
R² validation   : 0.9805


In [21]:
boosting_model_pv, pred_valid_boosting_pv = evaluate_model(
    target_name="Production photovoltaïque",
    model_name="HistGradientBoosting",
    model=HistGradientBoostingRegressor(
        random_state=42
    ),
    X_train_data=X_train,
    X_validation_data=X_valid,
    y_train_data=y_train_pv,
    y_validation_data=y_valid_pv
)

Cible   : Production photovoltaïque
Modèle  : HistGradientBoosting
-----------------------------------------------------------------
MAE validation  : 337.66
RMSE validation : 556.89
R² validation   : 0.9879


# 15. Comparaison des modèles pour la production éolienne

Les deux modèles sont appliqués à la prévision de la production éolienne à une
heure.

Cette cible peut être plus difficile à prévoir que la production
photovoltaïque en raison de la variabilité du vent. La comparaison sur
l'ensemble de validation permettra d'évaluer la capacité des modèles à
représenter cette dynamique.


In [22]:
pred_valid_naive_wind = evaluate_seasonal_naive(
    target_name="Production éolienne",
    full_target=y_wind,
    validation_index=y_valid_wind.index
)


Cible   : Production éolienne
Modèle  : Naïf saisonnier journalier
-----------------------------------------------------------------
MAE validation  : 992.96
RMSE validation : 1296.87
R² validation   : 0.0734


In [23]:
linear_model_wind, pred_valid_linear_wind = evaluate_model(
    target_name="Production éolienne",
    model_name="Régression linéaire",
    model=LinearRegression(),
    X_train_data=X_train_scaled,
    X_validation_data=X_valid_scaled,
    y_train_data=y_train_wind,
    y_validation_data=y_valid_wind
)

Cible   : Production éolienne
Modèle  : Régression linéaire
-----------------------------------------------------------------
MAE validation  : 197.87
RMSE validation : 264.13
R² validation   : 0.9616


In [24]:
boosting_model_wind, pred_valid_boosting_wind = evaluate_model(
    target_name="Production éolienne",
    model_name="HistGradientBoosting",
    model=HistGradientBoostingRegressor(
        random_state=42
    ),
    X_train_data=X_train,
    X_validation_data=X_valid,
    y_train_data=y_train_wind,
    y_validation_data=y_valid_wind
)

Cible   : Production éolienne
Modèle  : HistGradientBoosting
-----------------------------------------------------------------
MAE validation  : 205.30
RMSE validation : 273.75
R² validation   : 0.9587


# 16. Comparaison des modèles pour la demande électrique

Les modèles sont enfin appliqués à la prévision de la demande électrique à une
heure.

Cette variable dépend notamment des cycles horaires et hebdomadaires, des
conditions météorologiques et de l'historique récent de la consommation.


In [25]:
pred_valid_naive_demand = evaluate_seasonal_naive(
    target_name="Demande électrique",
    full_target=y_demand,
    validation_index=y_valid_demand.index
)


Cible   : Demande électrique
Modèle  : Naïf saisonnier journalier
-----------------------------------------------------------------
MAE validation  : 1242.41
RMSE validation : 1808.83
R² validation   : 0.8494


In [26]:
linear_model_demand, pred_valid_linear_demand = evaluate_model(
    target_name="Demande électrique",
    model_name="Régression linéaire",
    model=LinearRegression(),
    X_train_data=X_train_scaled,
    X_validation_data=X_valid_scaled,
    y_train_data=y_train_demand,
    y_validation_data=y_valid_demand
)

Cible   : Demande électrique
Modèle  : Régression linéaire
-----------------------------------------------------------------
MAE validation  : 262.39
RMSE validation : 344.32
R² validation   : 0.9945


In [27]:
boosting_model_demand, pred_valid_boosting_demand = evaluate_model(
    target_name="Demande électrique",
    model_name="HistGradientBoosting",
    model=HistGradientBoostingRegressor(
        random_state=42
    ),
    X_train_data=X_train,
    X_validation_data=X_valid,
    y_train_data=y_train_demand,
    y_validation_data=y_valid_demand
)

Cible   : Demande électrique
Modèle  : HistGradientBoosting
-----------------------------------------------------------------
MAE validation  : 215.12
RMSE validation : 285.65
R² validation   : 0.9962


In [28]:
# 16. TABLEAU COMPARATIF SUR LA VALIDATION

validation_results_df = pd.DataFrame(
    validation_results
)

validation_results_df = (
    validation_results_df
    .sort_values(
        by=["Cible", "RMSE_validation"]
    )
    .reset_index(drop=True)
)

display(validation_results_df)

,Cible,Modèle,MAE_validation,RMSE_validation,R2_validation
0,Demande électrique,HistGradientBoosting,215.120,285.645,0.996
1,Demande électrique,Régression linéaire,262.392,344.321,0.995
2,Demande électrique,Naïf saisonnier journalier,1242.410,1808.828,0.849
3,Production photovoltaïque,HistGradientBoosting,337.658,556.893,0.988
4,Production photovoltaïque,Régression linéaire,508.674,708.113,0.980
5,Production photovoltaïque,Naïf saisonnier journalier,523.482,1030.392,0.959
6,Production éolienne,Régression linéaire,197.868,264.133,0.962
7,Production éolienne,HistGradientBoosting,205.297,273.748,0.959
8,Production éolienne,Naïf saisonnier journalier,992.964,1296.872,0.073


## Gain des modèles par rapport à la baseline

Le gain est calculé à partir du RMSE. Une valeur positive indique que le modèle réduit l'erreur par rapport à la prévision naïve saisonnière.


In [29]:
# CALCUL DU GAIN PAR RAPPORT À LA BASELINE SAISONNIÈRE

baseline_rmse_validation = (
    validation_results_df[
        validation_results_df["Modèle"] == "Naïf saisonnier journalier"
    ]
    .set_index("Cible")["RMSE_validation"]
)

validation_results_df["Gain_RMSE_vs_naif_%"] = validation_results_df.apply(
    lambda row: (
        100
        * (
            baseline_rmse_validation.loc[row["Cible"]]
            - row["RMSE_validation"]
        )
        / baseline_rmse_validation.loc[row["Cible"]]
    ),
    axis=1
)

validation_results_df.loc[
    validation_results_df["Modèle"] == "Naïf saisonnier journalier",
    "Gain_RMSE_vs_naif_%"
] = 0.0

display(
    validation_results_df.sort_values(
        by=["Cible", "RMSE_validation"]
    ).reset_index(drop=True)
)


,Cible,Modèle,MAE_validation,RMSE_validation,R2_validation,Gain_RMSE_vs_naif_%
0,Demande électrique,HistGradientBoosting,215.120,285.645,0.996,84.208
1,Demande électrique,Régression linéaire,262.392,344.321,0.995,80.964
2,Demande électrique,Naïf saisonnier journalier,1242.410,1808.828,0.849,0.000
3,Production photovoltaïque,HistGradientBoosting,337.658,556.893,0.988,45.953
4,Production photovoltaïque,Régression linéaire,508.674,708.113,0.980,31.277
5,Production photovoltaïque,Naïf saisonnier journalier,523.482,1030.392,0.959,0.000
6,Production éolienne,Régression linéaire,197.868,264.133,0.962,79.633
7,Production éolienne,HistGradientBoosting,205.297,273.748,0.959,78.892
8,Production éolienne,Naïf saisonnier journalier,992.964,1296.872,0.073,0.000


# 17. Analyse comparative des modèles

Le tableau précédent met en évidence des différences de performances selon la
variable cible considérée.

Pour la production photovoltaïque et la demande électrique, le modèle
HistGradientBoosting obtient les meilleures performances en réduisant les
erreurs de prédiction et en augmentant le coefficient de détermination.

En revanche, pour la production éolienne, la régression linéaire obtient des
résultats légèrement supérieurs à ceux du modèle HistGradientBoosting sur
l'ensemble de validation.

Ces observations montrent que le choix du modèle dépend des caractéristiques
propres à chaque variable cible et qu'un modèle plus complexe n'est pas
nécessairement le plus performant.


# 18. Sauvegarde des résultats

Les performances obtenues sur l'ensemble de validation sont sauvegardées afin
de garantir la reproductibilité des expériences et de faciliter leur
utilisation dans les analyses ultérieures.


In [30]:
# 18. SAUVEGARDE DES RÉSULTATS

RESULTS_FILE = RESULTS_DIR / "validation_results.csv"

validation_results_df.to_csv(
    RESULTS_FILE,
    index=False
)

print("Résultats enregistrés.")
print("Fichier :", RESULTS_FILE)

Résultats enregistrés.
Fichier : c:\Users\celes\Desktop\PFE\results\validation_results.csv


# 19. Sauvegarde des modèles

Les modèles entraînés sont enregistrés afin de pouvoir être réutilisés sans
avoir à être réentraînés.

Cette étape facilitera leur utilisation dans le dashboard interactif développé
dans le dernier notebook du projet.


In [31]:
# 19. SAUVEGARDE DES MODÈLES

joblib.dump(
    linear_model_pv,
    MODELS_DIR / "linear_model_pv.pkl"
)

joblib.dump(
    boosting_model_pv,
    MODELS_DIR / "boosting_model_pv.pkl"
)

joblib.dump(
    linear_model_wind,
    MODELS_DIR / "linear_model_wind.pkl"
)

joblib.dump(
    boosting_model_wind,
    MODELS_DIR / "boosting_model_wind.pkl"
)

joblib.dump(
    linear_model_demand,
    MODELS_DIR / "linear_model_demand.pkl"
)

joblib.dump(
    boosting_model_demand,
    MODELS_DIR / "boosting_model_demand.pkl"
)

print("Tous les modèles sont enregistrés.")

Tous les modèles sont enregistrés.


# Conclusion du Notebook 05

Ce notebook a permis de construire les premiers modèles de prévision
énergétique à partir du jeu de données enrichi obtenu lors du Feature
Engineering.

Les principales étapes réalisées sont :

- sélection des variables explicatives et des variables cibles ;
- séparation chronologique des données en ensembles d'entraînement,
  de validation et de test ;
- normalisation des variables lorsque nécessaire ;
- entraînement de deux modèles de régression ;
- évaluation des performances sur l'ensemble de validation ;
- comparaison des modèles selon les métriques MAE, RMSE et R² ;
- sauvegarde des résultats et des modèles entraînés.

Les résultats obtenus montrent que HistGradientBoosting constitue le meilleur
modèle pour la production photovoltaïque et la demande électrique, tandis que
la régression linéaire obtient les meilleures performances pour la production
éolienne.

Les modèles sélectionnés à partir des performances observées sur l'ensemble de
validation seront évalués sur l'ensemble de test dans le Notebook 06.

Cette étape permettra d'obtenir une estimation finale de leur capacité de
généralisation sur des données jamais utilisées lors de l'entraînement ou de la
sélection des modèles.
